# DataBase Creation

In [2]:
import sqlite3
import json


def create_tables(conn):
    cursor = conn.cursor()

    # Table for recommendations (from Recommedations_(dynamic_version).json)
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS recommendations (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        category TEXT NOT NULL,
        item TEXT NOT NULL
    )
    """
    )

    # Table for coding problems (from Redirecting_Coding_Problem_(dynamic_version).json)
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS coding_problems (
        id TEXT PRIMARY KEY,
        title TEXT,
        difficulty TEXT,
        problem_description TEXT,
        problem_solution TEXT,
        link TEXT,
        saved BOOLEAN,
        completed BOOLEAN
    )
    """
    )

    # Table for test cases associated with coding problems
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS coding_problem_test_cases (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        problem_id TEXT,
        test_input TEXT,
        expected_output TEXT,
        FOREIGN KEY (problem_id) REFERENCES coding_problems(id)
    )
    """
    )

    # Table for learn page entries (from Redirecting_learn_page_(dynamic_version).json)
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS learn_page (
        id TEXT PRIMARY KEY,
        title TEXT,
        saved BOOLEAN,
        completed BOOLEAN
    )
    """
    )

    # Table for course files associated with each learn page entry
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS learn_courses (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        learn_id TEXT,
        course TEXT,
        FOREIGN KEY (learn_id) REFERENCES learn_page(id)
    )
    """
    )

    # Table for tests associated with each learn page entry
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS learn_tests (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        learn_id TEXT,
        test_id TEXT,
        file_path TEXT,
        score INTEGER,
        FOREIGN KEY (learn_id) REFERENCES learn_page(id)
    )
    """
    )

    # Table for MCQ tests (from Redirecting_MCQ_test_(dynamic_version).json)
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS mcq_tests (
        id TEXT PRIMARY KEY,
        title TEXT,
        test_file TEXT,
        saved BOOLEAN,
        completed BOOLEAN,
        score INTEGER
    )
    """
    )

    # Table for website user data (from User_data.json -> ForWebsite)
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS website_user (
        username TEXT PRIMARY KEY,
        password TEXT,
        profile_picture TEXT
    )
    """
    )

    # Table for resume personal information (from User_data.json -> ResumeDetails -> personalInformation)
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS resume_personal_information (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        first_name TEXT,
        last_name TEXT,
        email TEXT,
        city TEXT,
        state TEXT,
        country TEXT,
        linkedin TEXT,
        github TEXT
    )
    """
    )

    # Table for resume summary (from User_data.json -> ResumeDetails -> summary)
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS resume_summary (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        summary TEXT
    )
    """
    )

    # Table for resume experience (from User_data.json -> ResumeDetails -> experience)
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS resume_experience (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        company TEXT,
        position TEXT,
        start_date TEXT,
        end_date TEXT,
        responsibilities TEXT,
        achievements TEXT
    )
    """
    )

    # Table for resume education (from User_data.json -> ResumeDetails -> education)
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS resume_education (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        institution TEXT,
        degree TEXT,
        major TEXT,
        graduation_date TEXT
    )
    """
    )

    # Table for resume skills (from User_data.json -> ResumeDetails -> skills)
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS resume_skills (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        skill TEXT
    )
    """
    )

    # Table for resume projects (from User_data.json -> ResumeDetails -> projects)
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS resume_projects (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        description TEXT,
        technologies TEXT
    )
    """
    )

    # Table for resume awards and recognitions (from User_data.json -> ResumeDetails -> awardsAndRecognition)
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS resume_awards (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        award TEXT
    )
    """
    )

    conn.commit()


def insert_recommendations(cursor, data):
    # Data example: {"Learn_Page": ["LEPA_1", "LEPA_2", ...], "MCQs_page": [...], "coding_problems_page": [...]}
    for category, items in data.items():
        for item in items:
            cursor.execute(
                "INSERT INTO recommendations (category, item) VALUES (?, ?)",
                (category, item),
            )


def insert_coding_problems(cursor, data):
    # Data structure: { "problems": { "COPA_1": { ... }, "COPA_2": {...}, ... } }
    problems = data.get("problems", {})
    for problem_id, details in problems.items():
        cursor.execute(
            """
            INSERT INTO coding_problems (id, title, difficulty, problem_description, problem_solution, link, saved, completed)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
            (
                problem_id,
                details.get("title"),
                details.get("Difficulty"),
                details.get("Problem_Description"),
                details.get("Problem_Solution"),
                details.get("Link"),
                details.get("Saved"),
                details.get("Completed"),
            ),
        )
        # Insert test cases for each coding problem
        test_cases = details.get("Test_Cases", [])
        for test_case in test_cases:
            if len(test_case) == 2:
                cursor.execute(
                    """
                    INSERT INTO coding_problem_test_cases (problem_id, test_input, expected_output)
                    VALUES (?, ?, ?)
                """,
                    (problem_id, test_case[0], test_case[1]),
                )


def insert_learn_page(cursor, data):
    # Data is an array of objects
    for entry in data:
        learn_id = entry.get("ID")
        title = entry.get("title")
        saved = entry.get("Saved")
        completed = entry.get("Completed")
        cursor.execute(
            """
            INSERT INTO learn_page (id, title, saved, completed)
            VALUES (?, ?, ?, ?)
        """,
            (learn_id, title, saved, completed),
        )
        # Insert course files
        courses = entry.get("Course", [])
        for course in courses:
            cursor.execute(
                """
                INSERT INTO learn_courses (learn_id, course)
                VALUES (?, ?)
            """,
                (learn_id, course),
            )
        # Insert tests
        tests = entry.get("Test", [])
        for test in tests:
            test_id = test.get("id")
            file_path = test.get("File_path")
            score = test.get("score")
            cursor.execute(
                """
                INSERT INTO learn_tests (learn_id, test_id, file_path, score)
                VALUES (?, ?, ?, ?)
            """,
                (learn_id, test_id, file_path, score),
            )


def insert_mcq_tests(cursor, data):
    # Data is a dictionary where keys like "MCPA_1" map to MCQ test details
    for mcq_id, details in data.items():
        cursor.execute(
            """
            INSERT INTO mcq_tests (id, title, test_file, saved, completed, score)
            VALUES (?, ?, ?, ?, ?, ?)
        """,
            (
                mcq_id,
                details.get("Title"),
                details.get("Test_file"),
                details.get("Saved"),
                details.get("Completed"),
                details.get("Score"),
            ),
        )


def insert_user_data(cursor, data):
    # Insert website user data (from "ForWebsite")
    website_data = data.get("ForWebsite", {})
    username = website_data.get("username")
    password = website_data.get("password")
    profile_picture = website_data.get("profilePicture")
    cursor.execute(
        """
        INSERT INTO website_user (username, password, profile_picture)
        VALUES (?, ?, ?)
    """,
        (username, password, profile_picture),
    )

    # Insert resume details (from "ResumeDetails")
    resume = data.get("ResumeDetails", {})

    # Personal Information
    personal_info = resume.get("personalInformation", {})
    cursor.execute(
        """
        INSERT INTO resume_personal_information (first_name, last_name, email, city, state, country, linkedin, github)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """,
        (
            personal_info.get("firstName"),
            personal_info.get("lastName"),
            personal_info.get("email"),
            personal_info.get("location", {}).get("city"),
            personal_info.get("location", {}).get("state"),
            personal_info.get("location", {}).get("country"),
            personal_info.get("linkedin"),
            personal_info.get("github"),
        ),
    )

    # Summary
    summary = resume.get("summary", "")
    cursor.execute(
        """
        INSERT INTO resume_summary (summary)
        VALUES (?)
    """,
        (summary,),
    )

    # Experience (storing lists as JSON strings)
    experiences = resume.get("experience", [])
    for exp in experiences:
        responsibilities = json.dumps(exp.get("responsibilities", []))
        achievements = json.dumps(exp.get("achievements", []))
        cursor.execute(
            """
            INSERT INTO resume_experience (company, position, start_date, end_date, responsibilities, achievements)
            VALUES (?, ?, ?, ?, ?, ?)
        """,
            (
                exp.get("company"),
                exp.get("position"),
                exp.get("startDate"),
                exp.get("endDate"),
                responsibilities,
                achievements,
            ),
        )

    # Education
    educations = resume.get("education", [])
    for edu in educations:
        cursor.execute(
            """
            INSERT INTO resume_education (institution, degree, major, graduation_date)
            VALUES (?, ?, ?, ?)
        """,
            (
                edu.get("institution"),
                edu.get("degree"),
                edu.get("major"),
                edu.get("graduationDate"),
            ),
        )

    # Skills
    skills = resume.get("skills", [])
    for skill in skills:
        cursor.execute(
            """
            INSERT INTO resume_skills (skill)
            VALUES (?)
        """,
            (skill,),
        )

    # Projects (technologies stored as JSON string)
    for proj in resume.get("projects", []):
        technologies = json.dumps(proj.get("technologies", []))
        cursor.execute(
            """
            INSERT INTO resume_projects (name, description, technologies)
            VALUES (?, ?, ?)
        """,
            (proj.get("name"), proj.get("description"), technologies),
        )

    # Awards and Recognitions
    awards = resume.get("awardsAndRecognition", [])
    for award in awards:
        cursor.execute(
            """
            INSERT INTO resume_awards (award)
            VALUES (?)
        """,
            (award,),
        )


def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)


def main():
    # Connect to (or create) the SQLite database
    conn = sqlite3.connect("project.db")
    create_tables(conn)
    cursor = conn.cursor()

    # Load and insert data from each JSON file
    try:
        recommendations_data = load_json(
            r"Database\Redirection\Recommedations_(dynamic_version).json"
        )
        insert_recommendations(cursor, recommendations_data)
    except Exception as e:
        print("Error loading recommendations:", e)

    try:
        coding_problems_data = load_json(
            r"Database\Redirection\Redirecting_Coding_Problem_(dynamic_version).json"
        )
        insert_coding_problems(cursor, coding_problems_data)
    except Exception as e:
        print("Error loading coding problems:", e)

    try:
        learn_page_data = load_json(
            r"Database\Redirection\Redirecting_learn_page_(dynamic_version).json"
        )
        insert_learn_page(cursor, learn_page_data)
    except Exception as e:
        print("Error loading learn page data:", e)

    try:
        mcq_tests_data = load_json(
            r"Database\Redirection\Redirecting_MCQ_test_(dynamic_version).json"
        )
        insert_mcq_tests(cursor, mcq_tests_data)
    except Exception as e:
        print("Error loading MCQ tests:", e)

    try:
        user_data = load_json(r"Database\Redirection\User_data.json")
        insert_user_data(cursor, user_data)
    except Exception as e:
        print("Error loading user data:", e)

    conn.commit()
    conn.close()
    print("Database tables created and data inserted successfully.")


if __name__ == "__main__":
    main()

Database tables created and data inserted successfully.
